In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Đọc các file csv (đảm bảo file nằm cùng thư mục code)
orders = pd.read_csv('olist_orders_dataset_clean.csv')
items = pd.read_csv('olist_order_items_dataset_clean.csv')
payments = pd.read_csv('olist_order_payments_dataset_clean.csv')
products = pd.read_csv('olist_products_dataset_clean.csv')
reviews = pd.read_csv('olist_order_reviews_dataset_clean.csv')
customers=pd.read_csv('olist_customers_dataset_clean.csv')

In [20]:
# 2. Chuyển đổi thời gian 
date_cols = [
    'order_purchase_timestamp', 'order_approved_at', 
    'order_delivered_carrier_date', 'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]
for col in date_cols:
    # errors='coerce' sẽ biến các giá trị lỗi như "0" thành NaT
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# 3. Lọc đơn hàng đã giao (để đảm bảo có đủ ngày nhận hàng)
df_master = orders[orders['order_status'] == 'delivered'].copy()

In [21]:
# Độ trễ giao hàng (ngày): Dương là trễ, âm là sớm
df_master['raw_delivery_delay'] = (df_master['order_delivered_customer_date'] - df_master['order_estimated_delivery_date']).dt.total_seconds() / 86400

# Tổng thời gian chờ (ngày)
df_master['raw_lead_time'] = (df_master['order_delivered_customer_date'] - df_master['order_purchase_timestamp']).dt.total_seconds() / 86400

# Tốc độ chuẩn bị hàng của Seller (ngày)
df_master['raw_seller_speed'] = (df_master['order_delivered_carrier_date'] - df_master['order_approved_at']).dt.total_seconds() / 86400

In [22]:
# Tính Freight Ratio (Tỷ lệ phí ship)
item_agg = items.groupby('order_id').agg({'price': 'sum', 'freight_value': 'sum'}).reset_index()
item_agg['raw_freight_ratio'] = item_agg['freight_value'] / (item_agg['price'] + 0.001)

# Lấy số kỳ trả góp lớn nhất của đơn hàng
pay_agg = payments.groupby('order_id').agg({'payment_installments': 'max'}).reset_index()
pay_agg.rename(columns={'payment_installments': 'raw_installments'}, inplace=True)

# Lấy thông tin mô tả và ảnh sản phẩm (tính trung bình nếu đơn có nhiều món)
order_items_prod = items.merge(products, on='product_id')
prod_agg = order_items_prod.groupby('order_id').agg({
    'product_description_lenght': 'mean',
    'product_photos_qty': 'mean'
}).reset_index()
prod_agg.rename(columns={'product_description_lenght': 'raw_desc_length', 'product_photos_qty': 'raw_photos_qty'}, inplace=True)

In [23]:
# Merge tất cả vào df_master
df_final = df_master[['order_id', 'raw_delivery_delay', 'raw_lead_time', 'raw_seller_speed']]
df_final = df_final.merge(item_agg[['order_id', 'raw_freight_ratio']], on='order_id', how='left')
df_final = df_final.merge(pay_agg, on='order_id', how='left')
df_final = df_final.merge(prod_agg, on='order_id', how='left')

# Xử lý Outliers (Clipping) để thang đo 0-1 không bị méo
df_final['raw_delivery_delay'] = df_final['raw_delivery_delay'].clip(-30, 30) # Trễ/Sớm quá 30 ngày coi như mức trần
df_final['raw_freight_ratio'] = df_final['raw_freight_ratio'].clip(0, 2)     # Ship đắt gấp 2 lần hàng là tối đa
df_final.fillna(0, inplace=True) # Lấp đầy các giá trị thiếu bằng 0

In [24]:
price_agg = items.groupby('order_id')['price'].sum().reset_index()
price_agg.rename(columns={'price': 'raw_price'}, inplace=True)
df_final = df_final.merge(price_agg, on='order_id', how='left')
# 3. Xử lý giá trị ngoại lệ (Clipping) và giá trị thiếu
# Giới hạn giá trị ở mức 2000 BRL để các đơn hàng cực đắt không làm lệch thang đo
df_final['raw_price'] = df_final['raw_price'].clip(0, 2000)

# Điền giá trị 0 cho những trường hợp đơn hàng không có thông tin giá
df_final['raw_price'] = df_final['raw_price'].fillna(0)

# 4. Chuẩn hóa đặc trưng về thang đo [0, 1]
# Việc này giúp biến giá có cùng hệ quy chiếu với các biến thời gian và số lượng ảnh
scaler_price = MinMaxScaler()
df_final['norm_price'] = scaler_price.fit_transform(df_final[['raw_price']])

In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# merge core tables
df = orders.merge(items, on="order_id", how="inner") \
           .merge(reviews, on="order_id", how="inner") \
           .merge(products, on="product_id", how="left") \
           .merge(payments, on="order_id", how="left") \
           .merge(customers, on="customer_id", how="left")

In [26]:
#uy tín người bán hàng (seller reputation) - trung bình điểm đánh giá của các đơn hàng trước đó của seller
df = df.sort_values("review_creation_date")

df["seller_rep"] = (
    df.groupby("seller_id")["review_score"]
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

In [27]:
#số lần khách đã mua trước đó 
df = df.sort_values("order_purchase_timestamp")

df["customer_order_count"] = (
    df.groupby("customer_unique_id")
    .cumcount()
)

In [28]:
features_to_scale = [
    "seller_rep",
    "customer_order_count"
]

In [29]:
scaler = MinMaxScaler()

df[["norm_" + col for col in features_to_scale]] = scaler.fit_transform(
    df[features_to_scale]
)

In [30]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,payment_installments,payment_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,seller_rep,customer_order_count,norm_seller_rep,norm_customer_order_count
5292,2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,2016-10-18 13:14:51,NaT,2016-10-20,2,f293394c72c9b5fafd7023301fc21fc2,...,1.0,136.23,b7d76e111c89f7ebf14761390f0f7d17,69309,boa vista,RR,4.500000,0,0.875000,0.000000
5291,2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,2016-10-18 13:14:51,NaT,2016-10-20,1,c1488892604e4ba5cff5b4eb4d595400,...,1.0,136.23,b7d76e111c89f7ebf14761390f0f7d17,69309,boa vista,RR,3.333333,1,0.583333,0.013514
5131,e5fa5a7210941f7d56d0208e4e071d35,683c54fc24d40ee9f8a6fc179fd9856c,canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,NaT,NaT,2016-10-28,1,f3c2d01a84c947b078e32bbef0718962,...,3.0,75.06,4854e9b3feff728c13ee5fc7d1547e92,99025,passo fundo,RS,3.971831,0,0.742958,0.000000
36360,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,3,5a6b04657a4c5ee34285d1e4619a96b4,...,NaN,NaN,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,5.000000,0,1.000000,0.000000
36359,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,2,5a6b04657a4c5ee34285d1e4619a96b4,...,NaN,NaN,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,1.000000,1,0.000000,0.013514


In [31]:
#ghép thêm các đặc trưng đã tạo vào df_final
df_final = df_final.merge(df[["order_id", "norm_seller_rep", "norm_customer_order_count"]], on="order_id", how="left")

In [33]:
df_final.head()

,order_id,raw_delivery_delay,raw_lead_time,raw_seller_speed,raw_freight_ratio,raw_installments,raw_desc_length,raw_photos_qty,raw_price,norm_price,norm_seller_rep,norm_customer_order_count
0,e481f51cbdc54678b7cc49136f2d6af7,-7.107488,8.436574,2.366493,0.290754,1.0,268.0,4.0,29.99,0.014576,0.869318,0.013514
1,e481f51cbdc54678b7cc49136f2d6af7,-7.107488,8.436574,2.366493,0.290754,1.0,268.0,4.0,29.99,0.014576,0.872093,0.027027
2,e481f51cbdc54678b7cc49136f2d6af7,-7.107488,8.436574,2.366493,0.290754,1.0,268.0,4.0,29.99,0.014576,0.875000,0.040541
3,53cdb2fc8bc7dce0b6741e2150273451,-5.355729,13.782037,0.462882,0.191742,1.0,178.0,1.0,118.70,0.058950,0.894578,0.000000
4,47770eb9100c2d0c44946d9cf07ec65d,-17.245498,9.394213,0.204595,0.120199,3.0,232.0,1.0,159.90,0.079559,0.773610,0.000000


In [36]:
df_final[['order_id', 'norm_price', 'norm_seller_rep', 'norm_customer_order_count']].head()

,order_id,norm_price,norm_seller_rep,norm_customer_order_count
0,e481f51cbdc54678b7cc49136f2d6af7,0.014576,0.897436,0.043478
1,53cdb2fc8bc7dce0b6741e2150273451,0.058950,0.916667,0.000000
2,47770eb9100c2d0c44946d9cf07ec65d,0.079559,0.822665,0.000000
3,949d5b44dbf5de918fe9c16f97b45f8a,0.022084,0.868293,0.000000
4,ad21c59c0840e6cb83a9ceb5573f8159,0.009529,0.836000,0.000000
